
# Duplicate check for copyleft function corpora

This notebook checks:
1) exact duplicates (hash-based) within and across corpora files
2) whether duplicates appear with different license metadata (if present)
3) confirm suspected duplicates with Dolos via Docker


In [2]:
from pathlib import Path
import pandas as pd
import re
import hashlib
import shutil
import subprocess

DATA_DIR = Path("data")
assert DATA_DIR.exists(), f"Missing folder: {DATA_DIR.resolve()}"

CORPUS_FILES = sorted(DATA_DIR.glob("corpus_*_copyleft.jsonl"))
print("Found corpora:", len(CORPUS_FILES))
for p in CORPUS_FILES:
    print(" -", p.name)

assert len(CORPUS_FILES) > 0, "No files matched data/corpus_*_copyleft.jsonl"



Found corpora: 15
 - corpus_assembly_copyleft.jsonl
 - corpus_c_copyleft.jsonl
 - corpus_fortran_copyleft.jsonl
 - corpus_go_copyleft.jsonl
 - corpus_haskell_copyleft.jsonl
 - corpus_java_copyleft.jsonl
 - corpus_javascript_copyleft.jsonl
 - corpus_julia_copyleft.jsonl
 - corpus_lua_copyleft.jsonl
 - corpus_perl_copyleft.jsonl
 - corpus_python_copyleft.jsonl
 - corpus_ruby_copyleft.jsonl
 - corpus_rust_copyleft.jsonl
 - corpus_scala_copyleft.jsonl
 - corpus_sql_copyleft.jsonl


## Load all corpora

In [3]:
def load_jsonl(fp: Path) -> pd.DataFrame:
    df = pd.read_json(fp, lines=True)
    df["corpus_file"] = fp.name
    return df

dfs = []
for fp in CORPUS_FILES:
    try:
        dfs.append(load_jsonl(fp))
    except Exception as e:
        print(f"Failed reading {fp.name}: {e}")

df = pd.concat(dfs, ignore_index=True)
print("Total rows:", len(df))
print("Columns:", sorted(df.columns))
df.head(3)


Total rows: 2383422
Columns: ['code_normalized', 'code_original', 'corpus_file', 'filepath', 'language', 'license', 'n_lines', 'qualified_name', 'repo_dir', 'sha256_normalized', 'sha256_original', 'standalone']


,license,repo_dir,filepath,qualified_name,language,n_lines,sha256_original,sha256_normalized,code_original,code_normalized,standalone,corpus_file
0,GPL-3.0,01org_intel-hybrid-driver,src\shaders\utils\end_thread.asm,src::shaders::utils::end_thread.asm,assembly,30.0,52fb9722ef4125bf587757e1fbac6ba0f8cdc36c8e5395...,bbd6c6ec476ca20ae7536ca17e29ac21d933d330137140...,/*\n * Copyright © 2012 Intel Corporation\n *\...,/*\n * Copyright © 2012 Intel Corporation\n *\...,False,corpus_assembly_copyleft.jsonl
1,GPL-3.0,01org_iotg-lin-gfx-va-driver,src\shaders\h264\ildb\AVC_ILDB_Dep_Check.asm,src::shaders::h264::ildb::AVC_ILDB_Dep_Check.asm,assembly,399.0,879de968cf509371f66fc6b02bdc2a8ca329d88ab605c5...,038df7a4a22711e797fb87d98ef9c39a9e7aac327b9b85...,"/*\n\n * Copyright © <2010>, Intel Corporation...","/*\n\n * Copyright © <2010>, Intel Corporation...",False,corpus_assembly_copyleft.jsonl
2,GPL-3.0,01org_iotg-lin-gfx-va-driver,src\shaders\h264\ildb\load_Left_UV_2x8T.asm,src::shaders::h264::ildb::load_Left_UV_2x8T.asm,assembly,179.0,5b585730fcb282f0553c8d7bb2e6d0af24c240e7329897...,36141fcb00a0ae53069adf0bc37851902be8ea949eb45d...,"/*\n\n * Copyright © <2010>, Intel Corporation...","/*\n\n * Copyright © <2010>, Intel Corporation...",False,corpus_assembly_copyleft.jsonl


## Detect the code / language / license columns automatically

In [5]:
code_col = "code_normalized"   # or "code_original"
lang_col = "language"
name_col = "qualified_name"
lic_col  = None  # not present in these corpora

print("Using code_col:", code_col)
print("Using lang_col:", lang_col)
print("Using lic_col :", lic_col)
print("Using name_col:", name_col)


Using code_col: code_normalized
Using lang_col: language
Using lic_col : None
Using name_col: qualified_name


## Normalize + hash (exact duplicates)

In [8]:
hash_col = "sha256_normalized"

dups = (df.groupby(hash_col).size()
          .reset_index(name="count")
          .query("count >= 2")
          .sort_values("count", ascending=False))

dups.head(10)



,sha256_normalized,count
0,000002356dd2887d7a6d676ba45761599b3402f389bc02...,2
6,00005fe743618df519ca272e7947af9b3b206dbffb76c3...,2
7,000066d64d556919403966cfca91f6d541d04ef7bfbd2e...,2
17,00009fa113a0d1a1117e6382c02bbb247b6f7b9a63f2ec...,2
21,0000ecdb33384621182084e843722657c224e879e4d874...,2
26,00011e578ba9fa6da5f69877aef99d1740e1a3eb00b5f3...,2
27,000123b867eba83fb4d7be1de30e040660b377c96a5afc...,2
28,00012685ac0718d900d1757e6e9f685f5a2beb075cc119...,2
29,0001289a065a7a03ce70744412815ebef17d24595a89b5...,2
36,000164e221630eab65f991fac9c08d42b5fdabdda30738...,2


## Exact duplicates: how many?

In [9]:
hash_col = "sha256_normalized"

dup_hashes = set(dups[hash_col])
df_dup = df[df[hash_col].isin(dup_hashes)].copy()

print("Rows that belong to a duplicated hash:", len(df_dup))
print("Unique duplicate hashes:", df_dup[hash_col].nunique())


Rows that belong to a duplicated hash: 1229816
Unique duplicate hashes: 614908


## Are duplicates mostly within the same corpus file or across corpora?

In [10]:
by_file = (df_dup.groupby("corpus_file")
           .agg(
               duplicate_rows=(hash_col, "size"),
               duplicate_blocks=(hash_col, "nunique"),
           )
           .reset_index()
           .sort_values(["duplicate_blocks","duplicate_rows"], ascending=False))

by_file


,corpus_file,duplicate_rows,duplicate_blocks
0,corpus_java_copyleft.jsonl,941480,614908
1,corpus_ruby_copyleft.jsonl,288336,288336


## Are duplicates within the same file path (likely re-extraction) or across different paths?

In [11]:
# duplicates that appear in multiple different filepaths are more interesting
path_col = "filepath"  # from your screenshot

path_div = (df_dup.groupby(hash_col)[path_col]
            .nunique()
            .reset_index(name="n_distinct_filepaths")
            .sort_values("n_distinct_filepaths", ascending=False))

path_div.head(20)


,sha256_normalized,n_distinct_filepaths
0,000002356dd2887d7a6d676ba45761599b3402f389bc02...,1
1,00005fe743618df519ca272e7947af9b3b206dbffb76c3...,1
2,000066d64d556919403966cfca91f6d541d04ef7bfbd2e...,1
3,00009fa113a0d1a1117e6382c02bbb247b6f7b9a63f2ec...,1
4,0000ecdb33384621182084e843722657c224e879e4d874...,1
5,00011e578ba9fa6da5f69877aef99d1740e1a3eb00b5f3...,1
6,000123b867eba83fb4d7be1de30e040660b377c96a5afc...,1
7,00012685ac0718d900d1757e6e9f685f5a2beb075cc119...,1
8,0001289a065a7a03ce70744412815ebef17d24595a89b5...,1
9,000164e221630eab65f991fac9c08d42b5fdabdda30738...,1


In [12]:
top_hash = dups.iloc[0][hash_col]
rows = df[df[hash_col] == top_hash].copy()

cols = ["corpus_file", "language", "filepath", "qualified_name", "n_lines", "standalone"]
display(rows[cols])

print("\n--- code_normalized (first 60 lines) ---\n")
print("\n".join(rows.iloc[0]["code_normalized"].splitlines()[:60]))


,corpus_file,language,filepath,qualified_name,n_lines,standalone
553167,corpus_java_copyleft.jsonl,java,sample-service\src\main\java\com\xebia\samples...,sample-service::src::main::java::com::xebia::s...,21.0,False
1168075,corpus_java_copyleft.jsonl,java,sample-service\src\main\java\com\xebia\samples...,sample-service::src::main::java::com::xebia::s...,21.0,False



--- code_normalized (first 60 lines) ---

package com.xebia.sampleservice.hello;

import com.codahale.metrics.health.HealthCheck;

public class GreetingHealthCheck extends HealthCheck {
    private final String template;

    public GreetingHealthCheck(String template) {
        this.template = template;
    }

    @Override
    protected Result check() throws Exception {
        final String saying = String.format(template, "TEST");
        if (!saying.contains("TEST")) {
            return Result.unhealthy("template doesn't include a name");
        }
        return Result.healthy();
    }
}


In [13]:
# Deduplicate exact function copies (safe: same code + same filepath)
before = len(df)

df = (
    df.sort_values("filepath")  
      .drop_duplicates(subset=["sha256_normalized", "filepath"])
      .reset_index(drop=True)
)

after = len(df)

print(f"Rows before deduplication: {before:,}")
print(f"Rows after deduplication:  {after:,}")
print(f"Removed duplicates:       {before - after:,}")


Rows before deduplication: 2,383,422
Rows after deduplication:  1,768,514
Removed duplicates:       614,908


This notebook validated the integrity of the function-level copyleft corpora used in the experiment. Exact duplicates arising from repeated repository ingestion or re-extraction were identified using normalized code hashes and safely removed prior to sampling. The analysis confirms that remaining functions are distinct at the file-path level and that no systematic cross-corpus duplication affects the experimental setup.

After deduplication, the cleaned corpora were used as the basis for controlled sampling and code generation in subsequent notebooks. This preprocessing step reduces noise in the dataset and ensures that observed similarity between generated code and reference code reflects model behavior rather than artifacts of data duplication.